In [ ]:
from pathlib import Path
ENTITY = Path("../../data/entity").resolve()
train_file = str(ENTITY / "rad_graph_metric_train.json")
valid_file = str(ENTITY / "rad_graph_metric_validate.json")
test_file = str(ENTITY / "rad_graph_metric_test.json")
label_file = str(ENTITY / "landmark_observation_adj_mtx.npy")
disease_book = str(ENTITY / "observation_explanation.json")


In [ ]:
import json
train_ann = json.load(open(train_file,'r'))
train_img_path_list = list(train_ann)
train_set_list= list(set([path.replace(path.split('/')[-1],'') for path in train_img_path_list]))

val_ann = json.load(open(valid_file,'r'))
val_img_path_list = list(val_ann)
val_set_list= list(set([path.replace(path.split('/')[-1],'') for path in val_img_path_list]))

test_ann = json.load(open(test_file,'r'))
test_img_path_list = list(test_ann)
test_set_list= list(set([path.replace(path.split('/')[-1],'') for path in test_img_path_list]))

In [ ]:
import numpy as np
rad_graph_results = np.load(label_file)

In [ ]:
from medeha.constants import *
import pandas as pd
df = pd.read_csv(MIMIC_CXR_MASTER_CSV)

In [ ]:
from tqdm.auto import tqdm

basedir='/remote-home/share/medical/public/MIMIC-CXR-JPG/MIMIC-CXR/small'
# '/remote-home/share/medical/public/MIMIC-CXR-JPG/MIMIC-CXR/small/files/p10/p10000032/s50414267/02aa804e-bde0afdd-112c0b34-7bc16630-4e384014.jpg'
# ndf = pd.DataFrame(columns=['Path','text_path','text','Label_index','split'])
# ndf = pd.DataFrame(columns=['Path', 'dicom_id', 'subject_id', 'study_id', 'ViewPosition', 'impression', 'findings','text','Label_index', 'split'])

def getdf(ann,split):
    ndf = pd.DataFrame(columns=['Path', 'dicom_id', 'subject_id', 'study_id', 'ViewPosition', 'impression', 'findings', 'text','Label_index', 'split'])
    for i,(k,v) in enumerate(tqdm(ann.items())):
        k = k.replace(basedir,'2.0.0')
        df_row = df[df['Path'] == k]
        if len(df_row)==0:
            continue
        df_row = df_row.iloc[0]
        text=''
        text+= df_row['impression'] if df_row['impression'] != ' ' else ''
        text += " "
        text+= df_row['findings'] if df_row['findings'] != ' ' else ''
        text=text.replace('\n','')
        temp_df = pd.DataFrame([[k.replace('2.0.0',str(MIMIC_CXR_DATA_DIR)),df_row['dicom_id'],df_row['subject_id'],df_row['study_id'],df_row['ViewPosition'],df_row['impression'],df_row['findings'],text,v['labels_id'],split]],
                               columns=['Path', 'dicom_id', 'subject_id', 'study_id', 'ViewPosition', 'impression', 'findings', 'text','Label_index', 'split'])
        ndf = pd.concat([ndf, temp_df], ignore_index=True)
    return ndf

train_df = getdf(train_ann,"train")
val_df = getdf(val_ann,"valid")
test_df = getdf(test_ann,"test")

In [ ]:
ndf = pd.concat([train_df, val_df, test_df], ignore_index=True)
ndf.to_csv(MIMIC_CXR_MASTER_ENTITY_CSV)

In [ ]:
val_df

In [ ]:
ndf